# Compare CNN, LSTM, GRU, and random forest

Train each model on each 2, 4, and 6 second amplitude-window variant. The target is the window-center **collection phase** (preparation, action, resting); unknown labels are excluded. This is phase classification, not fall detection.

The selected sessions are grouped by session and attempt before splitting so overlapping windows and receiver views stay together. The test split measures held-out attempts, not participant- or session-independent generalization. Whole-recording min–max scaling and centered Hampel filtering use future samples, so this is an exploratory offline comparison.

Input windows come from `02_preprocess.ipynb`, which reads verified raw `.zip` session archives through the shared analysis loader. Training uses those derived windows and does not require legacy session directories.

Install the notebook dependencies with `uv sync --inexact --extra analysis --extra detection --extra notebook`, then select the project `.venv` kernel.

In [ ]:
import json
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import polars as pl
import torch
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch import nn
from torch.utils.data import DataLoader

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "fall_detection").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fall_detection.analysis.window_data import (
    LABELS,
    FileBatchSampler,
    WindowDataset,
    index_windows,
    save_manifest,
    split_attempts,
)

SESSION_IDS = [
    "20261104T081805521126Z",  # Khaelano
    "20261109T131546367822Z",  # Akmal
]

WINDOW_ROOT = PROJECT_ROOT / "data" / "interim" / "amplitude_windows_v1"
RESULT_DIR = PROJECT_ROOT / "data" / "experiments" / "phase_models_two_participants_v1"

SPLIT_SEED = 42
MAX_FILES_PER_VARIANT = None
WINDOW_DURATIONS = (2, 4, 6)
print("Sessions:", SESSION_IDS)
print("PyTorch device: CPU")
print("Results:", RESULT_DIR)

## Prepare one attempt-level split

Every model receives the same split. Each temporal bin contains the per-subcarrier mean of valid values and its observed fraction; empty bins use zero plus a zero mask. Pooling makes the input length consistent across window sizes. No additional scaler is fitted.

In [ ]:
combined_index = pl.concat(
    [
        index_windows(WINDOW_ROOT / session_id, MAX_FILES_PER_VARIANT)
        for session_id in SESSION_IDS
    ]
)

index = split_attempts(combined_index, SPLIT_SEED)
save_manifest(
    index,
    RESULT_DIR,
    {"split_seed": SPLIT_SEED, "max_files_per_variant": MAX_FILES_PER_VARIANT},
)
display(
    index.group_by(["duration", "split", "label"])
    .len()
    .sort(["duration", "split", "label"])
)
assert (
    index.group_by("group")
    .agg(pl.col("split").n_unique().alias("n"))
    .get_column("n")
    .max()
    == 1
)
print(
    f"Indexed {len(index):,} labeled windows across {index['group'].n_unique()} attempts."
)
print("Split assignments and dataset file fingerprints saved to", RESULT_DIR)

duration,split,label,len
i64,str,str,u32
2,"""test""","""action""",661
2,"""test""","""preparation""",594
2,"""test""","""resting""",597
2,"""train""","""action""",1880
2,"""train""","""preparation""",1746
…,…,…,…
6,"""test""","""resting""",100
6,"""train""","""action""",792
6,"""train""","""resting""",246


Indexed 14,526 labeled windows across 232 attempts.
Split assignments and dataset file fingerprints saved to /home/khaelano/Projects/radio-fall-detection/data/experiments/phase_models_v2/20261104T081805521126Z


## Train and evaluate

Each section below has independent settings and a run cell. Neural models select the checkpoint with the best validation macro-F1 and use the patience configured in their own section. The forest has its own settings. Running all models can take substantial time and memory.

In [ ]:
def metrics(y, predicted):
    """Compute fixed-label multiclass metrics, including absent test classes."""
    return {
        "accuracy": float(accuracy_score(y, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y, predicted)),
        "macro_f1": float(
            f1_score(
                y, predicted, labels=list(range(3)), average="macro", zero_division=0
            )
        ),
        "confusion_matrix": confusion_matrix(
            y, predicted, labels=list(range(3))
        ).tolist(),
        "classification_report": classification_report(
            y,
            predicted,
            labels=list(range(3)),
            target_names=LABELS,
            output_dict=True,
            zero_division=0,
        ),
    }


def _predict(model, loader, device):
    model.eval()
    ys, predictions = [], []
    with torch.no_grad():
        for x, y in loader:
            predictions.extend(model(x.to(device)).argmax(1).cpu().tolist())
            ys.extend(y.tolist())
    return np.asarray(ys), np.asarray(predictions)

## CNN

Configure and train the CNN separately for each window duration. The best validation macro-F1 selects the checkpoint.

In [4]:
CNN_CONFIG = {
    "seed": 42,
    "epochs": 12,
    "batch_size": 32,
    "learning_rate": 0.001,
    "patience": 3,
    "hidden_size": 64,
    "temporal_bins": 32,
}

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, features, hidden):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(features, hidden, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.head = nn.Linear(hidden, len(LABELS))

    def forward(self, x):
        return self.head(self.encoder(x.transpose(1, 2)))

In [ ]:
cnn_results = []
for duration in WINDOW_DURATIONS:
    print(f"Training CNN: {duration}s")
    random.seed(CNN_CONFIG["seed"])
    np.random.seed(CNN_CONFIG["seed"])
    torch.manual_seed(CNN_CONFIG["seed"])
    torch.use_deterministic_algorithms(True)
    subset = index.filter(pl.col("duration") == duration)
    datasets = {
        name: WindowDataset(
            subset.filter(pl.col("split") == name), CNN_CONFIG["temporal_bins"]
        )
        for name in ("train", "validation", "test")
    }
    if any(len(dataset) == 0 for dataset in datasets.values()):
        raise ValueError(f"Empty split for {duration}s")
    observed = {row[2] for row in datasets["train"].rows}
    if len(observed) < 2:
        raise ValueError(f"Fewer than two training classes: {observed}")
    output = RESULT_DIR / f"{duration}s" / "cnn"
    output.mkdir(parents=True, exist_ok=True)

    device = torch.device("cpu")
    loaders = {
        name: DataLoader(
            data, batch_size=CNN_CONFIG["batch_size"], shuffle=False, num_workers=0
        )
        for name, data in datasets.items()
        if name != "train"
    }
    loaders["train"] = DataLoader(
        datasets["train"],
        batch_sampler=FileBatchSampler(
            datasets["train"], CNN_CONFIG["batch_size"], CNN_CONFIG["seed"]
        ),
        num_workers=0,
    )
    features = datasets["train"][0][0].shape[1]
    model = CNNModel(features, CNN_CONFIG["hidden_size"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CNN_CONFIG["learning_rate"])
    loss_fn = nn.CrossEntropyLoss()
    best, stale, best_state, history = -1.0, 0, None, []
    for epoch in range(CNN_CONFIG["epochs"]):
        model.train()
        losses = []
        for x, y in loaders["train"]:
            optimizer.zero_grad()
            loss = loss_fn(model(x.to(device)), y.to(device))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        y_val, pred_val = _predict(model, loaders["validation"], device)
        score = metrics(y_val, pred_val)["macro_f1"]
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(losses)),
                "validation_macro_f1": score,
            }
        )
        if score > best:
            best, stale = score, 0
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
        else:
            stale += 1
            if stale >= CNN_CONFIG["patience"]:
                break
    model.load_state_dict(best_state)
    validation = metrics(*_predict(model, loaders["validation"], device))
    test = metrics(*_predict(model, loaders["test"], device))
    model_path = output / "model.pt"
    torch.save(
        {
            "state_dict": best_state,
            "kind": "cnn",
            "features": features,
            "hidden_size": CNN_CONFIG["hidden_size"],
            "labels": LABELS,
            "config": dict(CNN_CONFIG),
        },
        model_path,
    )
    result = {
        "duration_seconds": duration,
        "model": "cnn",
        "validation": validation,
        "test": test,
        "training_classes": sorted(observed),
        "absent_training_classes": sorted(set(LABELS) - observed),
        "counts": {key: len(value) for key, value in datasets.items()},
        "history": history,
        "config": dict(CNN_CONFIG),
    }
    (output / "metrics.json").write_text(json.dumps(result, indent=2))
    cnn_results.append(result)
    print(f"  Saved model: {model_path}")
    print(
        f"  validation macro-F1={validation['macro_f1']:.3f}; test macro-F1={test['macro_f1']:.3f}"
    )

## LSTM

Configure and train the LSTM separately for each window duration. The best validation macro-F1 selects the checkpoint.

In [7]:
LSTM_CONFIG = {
    "seed": 42,
    "epochs": 12,
    "batch_size": 32,
    "learning_rate": 0.001,
    "patience": 3,
    "hidden_size": 64,
    "temporal_bins": 32,
}

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, features, hidden):
        super().__init__()
        self.encoder = nn.LSTM(features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, len(LABELS))

    def forward(self, x):
        sequence, _ = self.encoder(x)
        return self.head(sequence[:, -1, :])

In [ ]:
lstm_results = []
for duration in WINDOW_DURATIONS:
    print(f"Training LSTM: {duration}s")
    random.seed(LSTM_CONFIG["seed"])
    np.random.seed(LSTM_CONFIG["seed"])
    torch.manual_seed(LSTM_CONFIG["seed"])
    torch.use_deterministic_algorithms(True)
    subset = index.filter(pl.col("duration") == duration)
    datasets = {
        name: WindowDataset(
            subset.filter(pl.col("split") == name), LSTM_CONFIG["temporal_bins"]
        )
        for name in ("train", "validation", "test")
    }
    if any(len(dataset) == 0 for dataset in datasets.values()):
        raise ValueError(f"Empty split for {duration}s")
    observed = {row[2] for row in datasets["train"].rows}
    if len(observed) < 2:
        raise ValueError(f"Fewer than two training classes: {observed}")
    output = RESULT_DIR / f"{duration}s" / "lstm"
    output.mkdir(parents=True, exist_ok=True)

    device = torch.device("cpu")
    loaders = {
        name: DataLoader(
            data, batch_size=LSTM_CONFIG["batch_size"], shuffle=False, num_workers=0
        )
        for name, data in datasets.items()
        if name != "train"
    }
    loaders["train"] = DataLoader(
        datasets["train"],
        batch_sampler=FileBatchSampler(
            datasets["train"], LSTM_CONFIG["batch_size"], LSTM_CONFIG["seed"]
        ),
        num_workers=0,
    )
    features = datasets["train"][0][0].shape[1]
    model = LSTMModel(features, LSTM_CONFIG["hidden_size"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_CONFIG["learning_rate"])
    loss_fn = nn.CrossEntropyLoss()
    best, stale, best_state, history = -1.0, 0, None, []
    for epoch in range(LSTM_CONFIG["epochs"]):
        model.train()
        losses = []
        for x, y in loaders["train"]:
            optimizer.zero_grad()
            loss = loss_fn(model(x.to(device)), y.to(device))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        y_val, pred_val = _predict(model, loaders["validation"], device)
        score = metrics(y_val, pred_val)["macro_f1"]
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(losses)),
                "validation_macro_f1": score,
            }
        )
        if score > best:
            best, stale = score, 0
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
        else:
            stale += 1
            if stale >= LSTM_CONFIG["patience"]:
                break
    model.load_state_dict(best_state)
    validation = metrics(*_predict(model, loaders["validation"], device))
    test = metrics(*_predict(model, loaders["test"], device))
    model_path = output / "model.pt"
    torch.save(
        {
            "state_dict": best_state,
            "kind": "lstm",
            "features": features,
            "hidden_size": LSTM_CONFIG["hidden_size"],
            "labels": LABELS,
            "config": dict(LSTM_CONFIG),
        },
        model_path,
    )
    result = {
        "duration_seconds": duration,
        "model": "lstm",
        "validation": validation,
        "test": test,
        "training_classes": sorted(observed),
        "absent_training_classes": sorted(set(LABELS) - observed),
        "counts": {key: len(value) for key, value in datasets.items()},
        "history": history,
        "config": dict(LSTM_CONFIG),
    }
    (output / "metrics.json").write_text(json.dumps(result, indent=2))
    lstm_results.append(result)
    print(f"  Saved model: {model_path}")
    print(
        f"  validation macro-F1={validation['macro_f1']:.3f}; test macro-F1={test['macro_f1']:.3f}"
    )

## GRU

Configure and train the GRU separately for each window duration. The best validation macro-F1 selects the checkpoint.

In [10]:
GRU_CONFIG = {
    "seed": 42,
    "epochs": 12,
    "batch_size": 32,
    "learning_rate": 0.001,
    "patience": 3,
    "hidden_size": 64,
    "temporal_bins": 32,
}

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, features, hidden):
        super().__init__()
        self.encoder = nn.GRU(features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, len(LABELS))

    def forward(self, x):
        sequence, _ = self.encoder(x)
        return self.head(sequence[:, -1, :])

In [ ]:
gru_results = []
for duration in WINDOW_DURATIONS:
    print(f"Training GRU: {duration}s")
    random.seed(GRU_CONFIG["seed"])
    np.random.seed(GRU_CONFIG["seed"])
    torch.manual_seed(GRU_CONFIG["seed"])
    torch.use_deterministic_algorithms(True)
    subset = index.filter(pl.col("duration") == duration)
    datasets = {
        name: WindowDataset(
            subset.filter(pl.col("split") == name), GRU_CONFIG["temporal_bins"]
        )
        for name in ("train", "validation", "test")
    }
    if any(len(dataset) == 0 for dataset in datasets.values()):
        raise ValueError(f"Empty split for {duration}s")
    observed = {row[2] for row in datasets["train"].rows}
    if len(observed) < 2:
        raise ValueError(f"Fewer than two training classes: {observed}")
    output = RESULT_DIR / f"{duration}s" / "gru"
    output.mkdir(parents=True, exist_ok=True)

    device = torch.device("cpu")
    loaders = {
        name: DataLoader(
            data, batch_size=GRU_CONFIG["batch_size"], shuffle=False, num_workers=0
        )
        for name, data in datasets.items()
        if name != "train"
    }
    loaders["train"] = DataLoader(
        datasets["train"],
        batch_sampler=FileBatchSampler(
            datasets["train"], GRU_CONFIG["batch_size"], GRU_CONFIG["seed"]
        ),
        num_workers=0,
    )
    features = datasets["train"][0][0].shape[1]
    model = GRUModel(features, GRU_CONFIG["hidden_size"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_CONFIG["learning_rate"])
    loss_fn = nn.CrossEntropyLoss()
    best, stale, best_state, history = -1.0, 0, None, []
    for epoch in range(GRU_CONFIG["epochs"]):
        model.train()
        losses = []
        for x, y in loaders["train"]:
            optimizer.zero_grad()
            loss = loss_fn(model(x.to(device)), y.to(device))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        y_val, pred_val = _predict(model, loaders["validation"], device)
        score = metrics(y_val, pred_val)["macro_f1"]
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(losses)),
                "validation_macro_f1": score,
            }
        )
        if score > best:
            best, stale = score, 0
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
        else:
            stale += 1
            if stale >= GRU_CONFIG["patience"]:
                break
    model.load_state_dict(best_state)
    validation = metrics(*_predict(model, loaders["validation"], device))
    test = metrics(*_predict(model, loaders["test"], device))
    model_path = output / "model.pt"
    torch.save(
        {
            "state_dict": best_state,
            "kind": "gru",
            "features": features,
            "hidden_size": GRU_CONFIG["hidden_size"],
            "labels": LABELS,
            "config": dict(GRU_CONFIG),
        },
        model_path,
    )
    result = {
        "duration_seconds": duration,
        "model": "gru",
        "validation": validation,
        "test": test,
        "training_classes": sorted(observed),
        "absent_training_classes": sorted(set(LABELS) - observed),
        "counts": {key: len(value) for key, value in datasets.items()},
        "history": history,
        "config": dict(GRU_CONFIG),
    }
    (output / "metrics.json").write_text(json.dumps(result, indent=2))
    gru_results.append(result)
    print(f"  Saved model: {model_path}")
    print(
        f"  validation macro-F1={validation['macro_f1']:.3f}; test macro-F1={test['macro_f1']:.3f}"
    )

## Random forest

Configure and fit the scikit-learn random forest independently for each window duration.

In [13]:
FOREST_CONFIG = {
    "seed": 42,
    "n_estimators": 150,
    "n_jobs": 2,
    "class_weight": "balanced_subsample",
    "temporal_bins": 32,
}

In [ ]:
forest_results = []
for duration in WINDOW_DURATIONS:
    print(f"Training random forest: {duration}s")
    subset = index.filter(pl.col("duration") == duration)
    datasets = {
        name: WindowDataset(
            subset.filter(pl.col("split") == name), FOREST_CONFIG["temporal_bins"]
        )
        for name in ("train", "validation", "test")
    }
    if any(len(dataset) == 0 for dataset in datasets.values()):
        raise ValueError(f"Empty split for {duration}s")
    observed = {row[2] for row in datasets["train"].rows}
    if len(observed) < 2:
        raise ValueError(f"Fewer than two training classes: {observed}")
    output = RESULT_DIR / f"{duration}s" / "random_forest"
    output.mkdir(parents=True, exist_ok=True)

    def materialize(dataset):
        xs, ys = zip(*(dataset[i] for i in range(len(dataset))))
        return np.stack([x.ravel() for x in xs]), np.asarray(ys)

    x_train, y_train = materialize(datasets["train"])
    model = RandomForestClassifier(
        n_estimators=FOREST_CONFIG["n_estimators"],
        random_state=FOREST_CONFIG["seed"],
        n_jobs=FOREST_CONFIG["n_jobs"],
        class_weight=FOREST_CONFIG["class_weight"],
    )
    model.fit(x_train, y_train)
    x_val, y_val = materialize(datasets["validation"])
    validation = metrics(y_val, model.predict(x_val))
    x_test, y_test = materialize(datasets["test"])
    test = metrics(y_test, model.predict(x_test))
    model_path = output / "model.joblib"
    joblib.dump(model, model_path)
    result = {
        "duration_seconds": duration,
        "model": "random_forest",
        "validation": validation,
        "test": test,
        "training_classes": sorted(observed),
        "absent_training_classes": sorted(set(LABELS) - observed),
        "counts": {key: len(value) for key, value in datasets.items()},
        "history": [],
        "config": dict(FOREST_CONFIG),
    }
    (output / "metrics.json").write_text(json.dumps(result, indent=2))
    forest_results.append(result)
    print(f"  Saved model: {model_path}")
    print(
        f"  validation macro-F1={validation['macro_f1']:.3f}; test macro-F1={test['macro_f1']:.3f}"
    )

## Compare results

Metrics describe classification of collection phases on held-out attempts from the selected sessions. Use the saved confusion matrices and per-class reports to inspect class imbalance and errors; do not interpret these results as fall-detection accuracy.

In [15]:
results = [
    result
    for name in ("cnn_results", "lstm_results", "gru_results", "forest_results")
    for result in globals().get(name, [])
]
if not results:
    raise ValueError("Run at least one model section before comparing results.")
comparison = pl.DataFrame(
    [
        {
            "window_seconds": result["duration_seconds"],
            "model": result["model"],
            "validation_macro_f1": result["validation"]["macro_f1"],
            "test_macro_f1": result["test"]["macro_f1"],
            "test_balanced_accuracy": result["test"]["balanced_accuracy"],
            "test_accuracy": result["test"]["accuracy"],
            "test_windows": result["counts"]["test"],
            "missing_training_classes": ", ".join(result["absent_training_classes"]),
        }
        for result in results
    ]
).sort(["window_seconds", "model"])
display(comparison)
comparison.write_csv(RESULT_DIR / "comparison.csv")

window_seconds,model,validation_macro_f1,test_macro_f1,test_balanced_accuracy,test_accuracy,test_windows,missing_training_classes
i64,str,f64,f64,f64,f64,i64,str
2,"""cnn""",0.559774,0.564782,0.57735,0.584233,1852,""""""
2,"""gru""",0.561149,0.564004,0.568683,0.575054,1852,""""""
2,"""lstm""",0.55318,0.56192,0.566363,0.574514,1852,""""""
2,"""random_forest""",0.458057,0.474568,0.481279,0.488661,1852,""""""
4,"""cnn""",0.302569,0.286948,0.357419,0.522388,737,""""""
…,…,…,…,…,…,…,…
4,"""random_forest""",0.351457,0.391852,0.429035,0.579376,737,""""""
6,"""cnn""",0.284963,0.281893,0.5,0.73262,374,"""preparation"""
6,"""gru""",0.284963,0.281893,0.5,0.73262,374,"""preparation"""


## Load a saved model

Each model and window size is saved under `RESULT_DIR/<duration>s/<model>/`: neural checkpoints use `model.pt` and random forests use `model.joblib`. The neural files contain weights and architecture settings; run the corresponding model-definition cell above before loading. Random forest files use joblib and should only be loaded from trusted sources.

Select a labeled test window below to check the saved model. The same mask-aware temporal pooling used during training is applied before inference.

In [ ]:
from fall_detection.analysis.window_data import pooled_input

LOAD_MODEL = "cnn"  # cnn, lstm, gru, random_forest
LOAD_WINDOW_SECONDS = 2  # 2, 4, or 6
LOAD_TEST_WINDOW_INDEX = 0

test_windows = index.filter(
    (pl.col("duration") == LOAD_WINDOW_SECONDS) & (pl.col("split") == "test")
)
sample = test_windows.row(LOAD_TEST_WINDOW_INDEX, named=True)
model_dir = RESULT_DIR / f"{LOAD_WINDOW_SECONDS}s" / LOAD_MODEL
with np.load(sample["path"], allow_pickle=False) as archive:
    sample_amplitude = archive["amplitude"][sample["index"]]
    sample_valid = archive["valid"][sample["index"]]
with (model_dir / "metrics.json").open() as stream:
    saved_config = json.load(stream)["config"]
x = pooled_input(sample_amplitude, sample_valid, saved_config["temporal_bins"])

if LOAD_MODEL == "random_forest":
    loaded_model = joblib.load(model_dir / "model.joblib")
    predicted_id = int(loaded_model.predict(x.reshape(1, -1))[0])
else:
    checkpoint = torch.load(
        model_dir / "model.pt", map_location="cpu", weights_only=True
    )
    if checkpoint["kind"] != LOAD_MODEL:
        raise ValueError("Checkpoint model kind does not match selection.")
    model_classes = {"cnn": "CNNModel", "lstm": "LSTMModel", "gru": "GRUModel"}
    class_name = model_classes[LOAD_MODEL]
    if class_name not in globals():
        raise RuntimeError(f"Run the {LOAD_MODEL.upper()} model-definition cell first.")
    loaded_model = globals()[class_name](
        checkpoint["features"], checkpoint["hidden_size"]
    )
    loaded_model.load_state_dict(checkpoint["state_dict"])
    loaded_model.eval()
    with torch.no_grad():
        predicted_id = int(
            loaded_model(torch.from_numpy(x).unsqueeze(0)).argmax(dim=1).item()
        )

print(f"Model: {model_dir}")
print(f"Actual phase: {sample['label']} | predicted phase: {LABELS[predicted_id]}")